# 03 Model 1: Word Embeddings + Classifier
Training of architecture 1: frozen pretrained word embeddings with a 2-layer classifier (ReLU non-linearity). Only the classifier is trained. Experiments are tracked with Weights & Biases.

**In short.** Question: how large must the classifier be, and which learning rate and batch size work best with the fixed-position input? Answer: all candidate runs that learned are close. Chosen: 600 hidden units, learning rate 0.000316, batch size 2048, best epoch 40 (validation loss 0.6460, accuracy 64.00%).

**GPU version (1.11):** run on a Tesla T4 on Google Colab, with deterministic mode, patience 10, the largest batch size that fits (limit 2048) and 5 seeds. The numbers come from this run, not from the CPU run on `main`. The cell that starts the sweep was not run again: its output is from the first GPU run, with the same code, and the cell after it evaluates the saved checkpoints.

In [1]:
# setup: on Colab, connect Google Drive; everywhere, use the GPU if there is one
import os
import sys

# on a GPU, a seed alone does not repeat a run exactly; this variable must be set before the GPU is used
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:2"

import torch

torch.backends.cudnn.deterministic = True
# deterministic operations where they exist, and a warning for the others
torch.use_deterministic_algorithms(True, warn_only=True)

if "google.colab" in sys.modules:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/nlp-project1-art")
    %pip install -q gensim datasets wandb nltk
    # the key comes from Colab Secrets, so it is not in the notebook
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
# the W&B server of this project
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

Mounted at /content/drive
cuda Tesla T4


Load the features and labels that were saved in `02_preprocessing.ipynb`. The test set is not loaded here: it is only evaluated once at the end, in `05_evaluation.ipynb`.

In [2]:
import numpy as np
import torch

data = np.load("features/model1.npz")
X_train = torch.tensor(data["X_train"]).to(device)
y_train = torch.tensor(data["y_train"]).to(device)
X_valid = torch.tensor(data["X_valid"]).to(device)
y_valid = torch.tensor(data["y_valid"]).to(device)
print(X_train.shape, X_train.dtype)
print(y_train.shape, y_train.dtype)

torch.Size([16801, 1200]) torch.float32
torch.Size([16801]) torch.int64


## Classifier
The 2-layer classifier like in the course exercise "PyTorch" (Week 1): linear layer, ReLU, linear layer. Input: 1200 features, output: 2 classes (h1 or h2).

In [3]:
import torch.nn as nn

class Classifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

torch.manual_seed(42)
model = Classifier(1200, 2, inner_dim=100).to(device)
print(model)

Classifier(
  (linear1): Linear(in_features=1200, out_features=100, bias=True)
  (relu): ReLU()
  (linear2): Linear(in_features=100, out_features=2, bias=True)
)


Number of parameters per layer (weights and biases), as in the course exercise "PyTorch" (Week 1).

In [4]:
for name, params in model.named_parameters():
    print(name, params.shape)

linear1.weight torch.Size([100, 1200])
linear1.bias torch.Size([100])
linear2.weight torch.Size([2, 100])
linear2.bias torch.Size([2])


Test the untrained classifier on the first 5 training examples: one pair of logits (h1, h2) per example.

In [5]:
x = X_train[:5]
output = model(x)
print(x.shape)
print(output.shape)
print(output)

torch.Size([5, 1200])
torch.Size([5, 2])
tensor([[-0.1848, -0.1143],
        [-0.1771, -0.0636],
        [-0.1528, -0.0544],
        [-0.1209, -0.0621],
        [-0.1661, -0.0300]], device='cuda:0', grad_fn=<AddmmBackward0>)


## Loss
Cross-entropy loss on 2 logits, one per hypothesis (course slides "Preliminaries", slide 27), as in the classifier of the course exercise "PyTorch"; the prediction is the hypothesis with the higher logit. 2 outputs with a softmax rather than 1 output with a sigmoid (course slides "Project Discussions" (Week 5), slide 22). `nn.CrossEntropyLoss` contains the softmax, so the classifier has no softmax layer ("double sigmoid" warning, same slides (Week 6), slide 30). Expected loss before training: about -log(0.5) ≈ 0.69.

In [6]:
criterion = nn.CrossEntropyLoss()

y = y_train[:5]
loss = criterion(output, y)
print(y)
print(loss)

tensor([1, 1, 0, 0, 1], device='cuda:0')
tensor(0.6781, device='cuda:0', grad_fn=<NllLossBackward0>)


## One training step
Adam (course slides "Preliminaries", slide 29) rather than AdamW (course slides "Project Discussions" (Week 6), slide 31): without weight decay, both make the same update steps (algorithm in the PyTorch documentation `torch.optim.AdamW`; Loshchilov & Hutter, 2019). Check: one training step on the 5 examples, as in the course exercise "PyTorch" (Week 1), must lower their loss.

In [7]:
from torch.optim import Adam

optimizer = Adam(model.parameters(), lr=0.001)

output = model(x)
loss = criterion(output, y)
optimizer.zero_grad()
loss.backward()
optimizer.step()

print(loss)
print(criterion(model(x), y))

tensor(0.6781, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6114, device='cuda:0', grad_fn=<NllLossBackward0>)


## Batches
The largest batch size that fits into the GPU memory (course slides "Project Discussions" (Week 4), slide 14): one forward and backward pass with the longest example, binary search if it does not fit. Limit 2048, because a batch as large as the training data would make only one update per epoch; on the Tesla T4, 2048 fits. 16801 = 8 x 2048 + 417, so `drop_last=True` skips 417 examples per epoch, other ones every time (shuffled). Loss and accuracy are measured on equal batches, so that the average of the batch losses is the average over all examples: validation 10 x 100, training 53 x 317.

In [8]:
from torch.utils.data import TensorDataset, DataLoader

# the largest batch size that fits into the GPU memory, up to the limit, as in the course slides "Project Discussions"
# (Week 4), slide 14: the longest training example, one forward and backward pass, binary search
def batch_fits(build_model, X, y, index, size):
    try:
        model = build_model()
        x = X[index].repeat(size, *[1] * (X.dim() - 1))
        criterion(model(x), y[:1].repeat(size)).backward()
        del model, x
        torch.cuda.empty_cache()
        return True
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return False

# limit: a batch as large as the training data would make only one update per epoch
def largest_batch_size(build_model, X, y, limit=2048):
    # on a CPU, memory is not the limit: the batch size of the course exercise
    if device.type != "cuda":
        return 32
    # the example with the most real tokens
    index = (X != 0).reshape(len(X), -1).sum(dim=1).argmax()
    if batch_fits(build_model, X, y, index, limit):
        print("Largest batch size:", limit, "(the limit)")
        return limit
    # low fits, high does not; multiples of 32 leave a little memory free
    low, high = 32, limit
    while high - low > 32:
        middle = (low + high) // 2 // 32 * 32
        if batch_fits(build_model, X, y, index, middle):
            low = middle
        else:
            high = middle
    print("Largest batch size:", low)
    return low

MAX_BATCH = largest_batch_size(lambda: Classifier(1200, 2, inner_dim=1000).to(device), X_train, y_train)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=MAX_BATCH, shuffle=True, drop_last=True)
train_eval_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=317)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=100)

x_batch, y_batch = next(iter(train_loader))
print(len(train_loader), len(train_eval_loader), len(valid_loader))
print(x_batch.shape, y_batch.shape)

Largest batch size: 2048 (the limit)
8 53 10
torch.Size([2048, 1200]) torch.Size([2048])


## Evaluation
Loss and accuracy as in the validation of the course exercise "Experiment Tracking"; `model.eval()` and `torch.no_grad()` as in the `test_loop` of the PyTorch tutorial "Optimizing Model Parameters". Check: an untrained model should reach about 0.69 loss and 50% accuracy.

In [9]:
def evaluate(model, loader):
    model.eval()
    losses, predictions, labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            output = model(x)
            losses.append(criterion(output, y).item())
            predictions.append(output.argmax(dim=1))
            labels.append(y)
    predictions = torch.cat(predictions)
    labels = torch.cat(labels)
    accuracy = (predictions == labels).float().mean().item()
    return np.mean(losses), accuracy

torch.manual_seed(42)
model = Classifier(1200, 2, inner_dim=100).to(device)
valid_loss, valid_acc = evaluate(model, valid_loader)
print(f"Validation loss: {valid_loss:.4f}, accuracy: {valid_acc:.2%}")

Validation loss: 0.6943, accuracy: 49.10%


## Experiment tracking
The runs are tracked with Weights and Biases (W&B), as in the course exercise "Experiment Tracking". The API key comes from Colab Secrets. `WANDB_SILENT` keeps the log lines of W&B out of the notebook output; the metrics are still logged.

In [10]:
import os

# no W&B log lines in the notebook output; the metrics are still logged
os.environ["WANDB_SILENT"] = "true"

import wandb
wandb.login()


/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


True

## Training
`train()` follows the PyTorch tutorial "Optimizing Model Parameters" and the course exercise "Experiment Tracking". Every call trains a fresh model with seed 42 and logs loss and accuracy per epoch to one W&B run. The cell ends with a walk-through run (10 epochs).

- **Checkpointing:** the `state_dict` of the epoch with the lowest validation loss, which changes more smoothly than the accuracy. The best values go into the W&B run summary, to sort the runs.
- **Early stopping:** after 10 epochs without a lower validation loss (course slides "Project Discussions" (Week 6), slide 38). The validation loss is noisy: with 300 hidden units and learning rate 0.0001, it is 0.6496 in epoch 63 and only lower again in epoch 70, so a patience of 5 would have stopped that run too early.
- **Learning rate:** constant, which can work well (same slides, slide 33, with Hägele et al., 2024).
- **Regularization:** no dropout or weight decay (same slides, slide 34). Selecting the best epoch stops before the validation loss rises, but a gap remains: the chosen run has a training accuracy of 77.32% and a validation accuracy of 64.00% at its best epoch (W&B).
- **Normalization:** none in the chosen configuration; tested in the section "Layer normalization".

In [11]:
import os

os.makedirs("checkpoints", exist_ok=True)

def train(inner_dim=100, learning_rate=0.001, num_epochs=10, patience=10):
    name = f"glove-mlp-h{inner_dim}-lr{learning_rate}-e{num_epochs}"
    checkpoint_path = f"checkpoints/{name}.pth"
    wandb.init(
        project="nlp-project1-art-gpu",
        name=name,
        config={
            "model": "glove-mlp",
            "inner_dim": inner_dim,
            "learning_rate": learning_rate,
            "max_epochs": num_epochs,
            "patience": patience,
            "batch_size": train_loader.batch_size,
            "seed": 42,
        },
    )

    torch.manual_seed(42)
    model = Classifier(1200, 2, inner_dim=inner_dim).to(device)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")
    # number of epochs in a row without a new lowest validation loss
    epochs_without_improvement = 0

    for epoch in range(num_epochs):
        model.train()
        for x, y in train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, train acc {train_acc:.2%}, valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            # early stopping: no new lowest validation loss for `patience` epochs in a row
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                print(f"Early stopping: no lower validation loss in the last {patience} epochs")
                break

    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    print(f"Best epoch: {best_epoch}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    return model

model = train(inner_dim=100, learning_rate=0.001)

Epoch 1: train loss 0.6902, train acc 56.24%, valid loss 0.6901, valid acc 56.00%
Epoch 2: train loss 0.6849, train acc 57.57%, valid loss 0.6868, valid acc 56.80%
Epoch 3: train loss 0.6774, train acc 59.60%, valid loss 0.6816, valid acc 56.90%
Epoch 4: train loss 0.6695, train acc 61.06%, valid loss 0.6758, valid acc 56.90%
Epoch 5: train loss 0.6624, train acc 61.41%, valid loss 0.6718, valid acc 57.50%
Epoch 6: train loss 0.6545, train acc 62.36%, valid loss 0.6677, valid acc 57.00%
Epoch 7: train loss 0.6515, train acc 61.39%, valid loss 0.6684, valid acc 58.80%
Epoch 8: train loss 0.6429, train acc 63.50%, valid loss 0.6630, valid acc 58.80%
Epoch 9: train loss 0.6391, train acc 63.62%, valid loss 0.6614, valid acc 59.40%
Epoch 10: train loss 0.6357, train acc 63.66%, valid loss 0.6620, valid acc 58.70%
Best epoch: 9, valid loss 0.6614, valid acc 59.40%


Check that the returned model is the one of the best epoch: its validation loss and accuracy must be the same as in that epoch.

In [12]:
valid_loss, valid_acc = evaluate(model, valid_loader)
print(f"Validation loss: {valid_loss:.4f}, accuracy: {valid_acc:.2%}")

Validation loss: 0.6614, accuracy: 59.40%


## Experiments
Hyperparameter tuning on the validation set (course slides "Course Project HS26", slide 10). Questions: how many hidden units does the classifier need, and which learning rate works, "the most important hyperparameter" (course slides "Project Discussions" (Week 6), slide 32)? A grid search (same slides (Week 7), slide 44) trains all 6 combinations of 50, 100 and 300 hidden units and learning rate 0.001 and 0.01, each for at most 20 epochs. With batch size 2048, an epoch has only 8 updates, so most runs are still improving at this limit; the section "Sweep" allows more epochs. The seed is set inside `train()`, so runs with the same number of hidden units start from the same weights and see the same batches. Each configuration is trained once: a run is exactly reproducible, but small differences between configurations can be chance.

In [13]:
for inner_dim in [50, 100, 300]:
    for learning_rate in [0.001, 0.01]:
        train(inner_dim=inner_dim, learning_rate=learning_rate, num_epochs=20)

Epoch 1: train loss 0.6904, train acc 55.44%, valid loss 0.6904, valid acc 56.70%
Epoch 2: train loss 0.6851, train acc 58.50%, valid loss 0.6867, valid acc 56.80%
Epoch 3: train loss 0.6790, train acc 58.66%, valid loss 0.6824, valid acc 57.80%
Epoch 4: train loss 0.6722, train acc 60.13%, valid loss 0.6783, valid acc 57.70%
Epoch 5: train loss 0.6654, train acc 61.15%, valid loss 0.6738, valid acc 57.50%
Epoch 6: train loss 0.6594, train acc 61.90%, valid loss 0.6701, valid acc 58.30%
Epoch 7: train loss 0.6555, train acc 61.65%, valid loss 0.6699, valid acc 58.20%
Epoch 8: train loss 0.6505, train acc 62.43%, valid loss 0.6667, valid acc 59.10%
Epoch 9: train loss 0.6463, train acc 62.75%, valid loss 0.6639, valid acc 57.70%
Epoch 10: train loss 0.6429, train acc 62.94%, valid loss 0.6633, valid acc 57.40%
Epoch 11: train loss 0.6376, train acc 63.76%, valid loss 0.6603, valid acc 59.30%
Epoch 12: train loss 0.6343, train acc 64.24%, valid loss 0.6582, valid acc 60.00%
Epoch 13: tra

## Learning rate 0.0001
The grid runs that learned end close together (0.6524 to 0.6539), and 0.01 collapses with 50 hidden units. So the next smaller value in exponential steps, 0.0001, is tested too (course slides "Project Discussions" (Week 7), slides 52 and 53), for 50, 100 and 300 hidden units, with at most 100 epochs because the steps are 10 times smaller (slide 41).

In [14]:
# the next smaller learning rate, with more epochs (smaller steps)
for inner_dim in [50, 100, 300]:
    train(inner_dim=inner_dim, learning_rate=0.0001, num_epochs=100)

Epoch 1: train loss 0.6930, train acc 50.85%, valid loss 0.6928, valid acc 50.90%
Epoch 2: train loss 0.6921, train acc 52.44%, valid loss 0.6920, valid acc 52.60%
Epoch 3: train loss 0.6913, train acc 53.05%, valid loss 0.6914, valid acc 53.30%
Epoch 4: train loss 0.6905, train acc 55.61%, valid loss 0.6909, valid acc 53.50%
Epoch 5: train loss 0.6895, train acc 56.53%, valid loss 0.6902, valid acc 53.90%
Epoch 6: train loss 0.6886, train acc 56.22%, valid loss 0.6895, valid acc 54.80%
Epoch 7: train loss 0.6877, train acc 57.76%, valid loss 0.6889, valid acc 54.30%
Epoch 8: train loss 0.6866, train acc 57.90%, valid loss 0.6881, valid acc 56.40%
Epoch 9: train loss 0.6856, train acc 58.87%, valid loss 0.6874, valid acc 56.70%
Epoch 10: train loss 0.6844, train acc 58.72%, valid loss 0.6866, valid acc 57.10%
Epoch 11: train loss 0.6833, train acc 59.60%, valid loss 0.6858, valid acc 56.50%
Epoch 12: train loss 0.6821, train acc 59.39%, valid loss 0.6849, valid acc 57.30%
Epoch 13: tra

## Extending the tested range: learning rate 0.00001 and 1000 hidden units
The best run so far, 300 hidden units with learning rate 0.0001, has the smallest learning rate and the most hidden units tested, so the next values are tested too (course slides "Project Discussions" (Week 7), slides 52 and 53), each with the other setting of the best run: learning rate 0.00001 (at most 300 epochs) and 1000 hidden units (at most 100 epochs).

The run with 100 hidden units and learning rate 0.01 has its best epoch at the last of its 20 epochs, so it is also trained for at most 100 epochs (slide 41).

In [15]:
# test the next values: a smaller learning rate and more hidden units;
# and more epochs for the run whose best epoch was its last epoch
for inner_dim, learning_rate, num_epochs in [(300, 0.00001, 300), (1000, 0.0001, 100), (100, 0.01, 100)]:
    train(inner_dim=inner_dim, learning_rate=learning_rate, num_epochs=num_epochs)

Epoch 1: train loss 0.6929, train acc 51.22%, valid loss 0.6931, valid acc 51.30%
Epoch 2: train loss 0.6927, train acc 51.33%, valid loss 0.6929, valid acc 51.70%
Epoch 3: train loss 0.6925, train acc 52.09%, valid loss 0.6928, valid acc 51.60%
Epoch 4: train loss 0.6924, train acc 52.56%, valid loss 0.6927, valid acc 53.20%
Epoch 5: train loss 0.6922, train acc 53.03%, valid loss 0.6925, valid acc 52.70%
Epoch 6: train loss 0.6920, train acc 53.40%, valid loss 0.6924, valid acc 52.80%
Epoch 7: train loss 0.6918, train acc 52.98%, valid loss 0.6922, valid acc 53.20%
Epoch 8: train loss 0.6916, train acc 53.82%, valid loss 0.6921, valid acc 53.20%
Epoch 9: train loss 0.6914, train acc 54.46%, valid loss 0.6920, valid acc 54.30%
Epoch 10: train loss 0.6912, train acc 54.74%, valid loss 0.6919, valid acc 54.10%
Epoch 11: train loss 0.6911, train acc 55.09%, valid loss 0.6918, valid acc 54.10%
Epoch 12: train loss 0.6909, train acc 55.51%, valid loss 0.6917, valid acc 54.80%
Epoch 13: tra

## Layer normalization
Does layer normalization lower the validation loss of the best configuration of the grid (300 hidden units, learning rate 0.0001, at most 100 epochs)? It is an option in the course slides "Project Discussions" (Week 6), slide 34, and an example experiment in (Week 7), slide 44. `nn.LayerNorm` goes between the first linear layer and the ReLU (Ba et al., 2016). The check below shows that both runs start from the same weights and get the same first batch. If this run has the lowest validation loss of all candidate runs, it becomes the chosen configuration. Every 10th epoch is printed; all are logged to W&B.

In [16]:
# the same classifier, with layer normalization between the first linear layer and the ReLU
class LayerNormClassifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        # new: normalizes the inner_dim values of each example
        self.norm = nn.LayerNorm(inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        # [batch, inner_dim]
        out = self.linear1(x)
        # new: layer normalization, same shape
        out = self.norm(out)
        # ReLU, same shape
        out = self.relu(out)
        # [batch, 2]: one logit per hypothesis
        out = self.linear2(out)
        return out


In [17]:
# same seed as in train(): the model with layer norm, then the first batch
torch.manual_seed(42)
model_ln = LayerNormClassifier(1200, 2, inner_dim=300).to(device)
x_ln, _ = next(iter(train_loader))

# the same for the model without layer norm
torch.manual_seed(42)
model_plain = Classifier(1200, 2, inner_dim=300).to(device)
x_plain, _ = next(iter(train_loader))

# same starting weights and same first batch?
print(torch.equal(model_ln.linear1.weight, model_plain.linear1.weight))
print(torch.equal(model_ln.linear2.weight, model_plain.linear2.weight))
print(torch.equal(x_ln, x_plain))
# how many parameters does layer norm add?
print(sum(p.numel() for p in model_ln.parameters()) - sum(p.numel() for p in model_plain.parameters()))


True
True
True
600


In [18]:
# the best configuration of the grid, with layer norm
inner_dim, learning_rate, num_epochs, patience = 300, 0.0001, 100, 10
name = f"glove-mlp-ln-h{inner_dim}-lr{learning_rate}-e{num_epochs}"
checkpoint_path = f"checkpoints/{name}.pth"
wandb.init(
    project="nlp-project1-art-gpu",
    name=name,
    config={
        "model": "glove-mlp-ln",
        "inner_dim": inner_dim,
        "learning_rate": learning_rate,
        "max_epochs": num_epochs,
        "patience": patience,
        "batch_size": train_loader.batch_size,
        "seed": 42,
    },
)

# same training as in train(), with the layer norm class
torch.manual_seed(42)
model = LayerNormClassifier(1200, 2, inner_dim=inner_dim).to(device)
optimizer = Adam(model.parameters(), lr=learning_rate)
best_valid_loss = float("inf")
# number of epochs in a row without a new lowest validation loss
epochs_without_improvement = 0

for epoch in range(num_epochs):
    model.train()
    for x, y in train_loader:
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss, train_acc = evaluate(model, train_eval_loader)
    valid_loss, valid_acc = evaluate(model, valid_loader)
    # print every 10th epoch (all epochs are in W&B)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, train acc {train_acc:.2%}, valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")
    wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_valid_acc = valid_acc
        best_epoch = epoch + 1
        torch.save(model.state_dict(), checkpoint_path)
        epochs_without_improvement = 0
    else:
        # early stopping: no new lowest validation loss for `patience` epochs in a row
        epochs_without_improvement += 1
        if epochs_without_improvement == patience:
            print(f"Early stopping: no lower validation loss in the last {patience} epochs")
            break

wandb.run.summary["best_val_loss"] = best_valid_loss
wandb.run.summary["best_val_acc"] = best_valid_acc
wandb.run.summary["best_epoch"] = best_epoch
wandb.finish()
print(f"Best epoch: {best_epoch}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")
print("For comparison, without layer norm (best run of the grid): best epoch 74, valid loss 0.6492, valid acc 62.30%")

Epoch 10: train loss 0.6095, train acc 67.63%, valid loss 0.6652, valid acc 58.60%
Epoch 20: train loss 0.5248, train acc 76.87%, valid loss 0.6566, valid acc 60.90%
Epoch 30: train loss 0.4464, train acc 84.16%, valid loss 0.6589, valid acc 61.80%
Early stopping: no lower validation loss in the last 10 epochs
Best epoch: 22, valid loss 0.6525, valid acc 62.10%
For comparison, without layer norm (best run of the grid): best epoch 74, valid loss 0.6492, valid acc 62.30%


## Results
Load the best checkpoint of every run so far (the sweep follows in its own section), evaluate it on the validation set, and sort the results by the validation loss (lowest first).

In [19]:
results = []
# all runs: hidden units, learning rate and number of epochs
runs = [(inner_dim, learning_rate, num_epochs) for inner_dim in [50, 100, 300]
        for learning_rate, num_epochs in [(0.0001, 100), (0.001, 20), (0.01, 20)]]
runs += [(300, 0.00001, 300), (1000, 0.0001, 100), (100, 0.01, 100)]

for inner_dim, learning_rate, num_epochs in runs:
    model = Classifier(1200, 2, inner_dim=inner_dim).to(device)
    model.load_state_dict(torch.load(f"checkpoints/glove-mlp-h{inner_dim}-lr{learning_rate}-e{num_epochs}.pth", weights_only=True))
    valid_loss, valid_acc = evaluate(model, valid_loader)
    results.append((valid_loss, valid_acc, inner_dim, learning_rate, num_epochs))

# the run with layer norm: the same evaluation, with its own class
model = LayerNormClassifier(1200, 2, inner_dim=300).to(device)
model.load_state_dict(torch.load("checkpoints/glove-mlp-ln-h300-lr0.0001-e100.pth", weights_only=True))
valid_loss, valid_acc = evaluate(model, valid_loader)
results.append((valid_loss, valid_acc, "300 with layer norm", 0.0001, 100))

for valid_loss, valid_acc, inner_dim, learning_rate, num_epochs in sorted(results):
    print(f"h{inner_dim}, lr {learning_rate}, {num_epochs} epochs: valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")

h300, lr 0.0001, 100 epochs: valid loss 0.6492, valid acc 62.30%
h1000, lr 0.0001, 100 epochs: valid loss 0.6492, valid acc 62.20%
h100, lr 0.0001, 100 epochs: valid loss 0.6506, valid acc 61.40%
h100, lr 0.01, 20 epochs: valid loss 0.6524, valid acc 61.20%
h100, lr 0.01, 100 epochs: valid loss 0.6524, valid acc 61.20%
h50, lr 0.0001, 100 epochs: valid loss 0.6525, valid acc 60.70%
h300 with layer norm, lr 0.0001, 100 epochs: valid loss 0.6525, valid acc 62.10%
h300, lr 0.001, 20 epochs: valid loss 0.6527, valid acc 62.00%
h100, lr 0.001, 20 epochs: valid loss 0.6530, valid acc 60.80%
h50, lr 0.001, 20 epochs: valid loss 0.6534, valid acc 60.80%
h300, lr 0.01, 20 epochs: valid loss 0.6539, valid acc 62.00%
h300, lr 1e-05, 300 epochs: valid loss 0.6567, valid acc 60.80%
h50, lr 0.01, 20 epochs: valid loss 0.6930, valid acc 50.80%


## Active ReLU units
With learning rate 0.01 and 50 hidden units, the model predicts the same class for every example. Count the units that output more than 0 for more than 1% of the training examples (169 or more of 16801), a practical threshold ("dying ReLU", Lu et al., 2020): before training, with the start weights of the runs (seed 42), and in the best checkpoints of the nine runs with 50, 100, 300 hidden units and learning rates 0.01, 0.001, 0.0001.

In [20]:
# no training here, so no gradients are needed
with torch.no_grad():
    for inner_dim in [50, 100, 300]:
        # same start weights as the training runs with this number of hidden units
        torch.manual_seed(42)
        untrained = Classifier(1200, 2, inner_dim=inner_dim).to(device)
        # hidden layer only (linear1 + ReLU): one row per example, one column per unit
        hidden = untrained.relu(untrained.linear1(X_train))
        # share of the examples on which each unit outputs more than 0; active if more than 1%
        active = (hidden > 0).float().mean(dim=0) > 0.01
        # count the active units (True counts as 1)
        print(f"h{inner_dim}, untrained: {active.sum().item()} of {inner_dim} units active")
        # the same for the trained models
        for learning_rate, num_epochs in [(0.0001, 100), (0.001, 20), (0.01, 20)]:
            # load the weights of the best epoch of the run (-e20 / -e100 = the run had 20 / 100 epochs)
            model = Classifier(1200, 2, inner_dim=inner_dim).to(device)
            model.load_state_dict(torch.load(f"checkpoints/glove-mlp-h{inner_dim}-lr{learning_rate}-e{num_epochs}.pth", weights_only=True))
            hidden = model.relu(model.linear1(X_train))
            active = (hidden > 0).float().mean(dim=0) > 0.01
            print(f"h{inner_dim}, lr {learning_rate}: {active.sum().item()} of {inner_dim} units active")

h50, untrained: 49 of 50 units active
h50, lr 0.0001: 50 of 50 units active
h50, lr 0.001: 28 of 50 units active
h50, lr 0.01: 0 of 50 units active
h100, untrained: 94 of 100 units active
h100, lr 0.0001: 100 of 100 units active
h100, lr 0.001: 62 of 100 units active
h100, lr 0.01: 3 of 100 units active
h300, untrained: 279 of 300 units active
h300, lr 0.0001: 300 of 300 units active
h300, lr 0.001: 142 of 300 units active
h300, lr 0.01: 8 of 300 units active


## Hypothesis-only check
How much of the result comes from the hypotheses alone? The best configuration of the grid is trained again with only h1 and h2 (the last 600 of the 1200 dimensions): a hypothesis-only baseline (Gururangan et al., 2018; Poliak et al., 2018; Bhagavatula et al., 2020, Table 1: BERT, 59.5% on the ART test set). It is a diagnostic, not a candidate: the task is to choose the hypothesis that explains the observations, so the input of model 1 stays all four texts. Every 10th epoch is printed; all are logged to W&B.

In [21]:
# only h1 and h2: the last 600 of the 1200 numbers (O1, O2, h1, h2, 300 each)
X_train_hyp = X_train[:, 600:]
X_valid_hyp = X_valid[:, 600:]
# the same batches as for the other runs
train_loader_hyp = DataLoader(TensorDataset(X_train_hyp, y_train), batch_size=MAX_BATCH, shuffle=True, drop_last=True)
train_eval_loader_hyp = DataLoader(TensorDataset(X_train_hyp, y_train), batch_size=317)
valid_loader_hyp = DataLoader(TensorDataset(X_valid_hyp, y_valid), batch_size=100)

# the best configuration of the grid
inner_dim, learning_rate, num_epochs, patience = 300, 0.0001, 100, 10
name = f"glove-mlp-hyp-only-h{inner_dim}-lr{learning_rate}-e{num_epochs}"
checkpoint_path = f"checkpoints/{name}.pth"
wandb.init(
    project="nlp-project1-art-gpu",
    name=name,
    config={
        "model": "glove-mlp-hyp-only",
        "inner_dim": inner_dim,
        "learning_rate": learning_rate,
        "max_epochs": num_epochs,
        "patience": patience,
        "batch_size": train_loader_hyp.batch_size,
        "seed": 42,
    },
)

# same training as in train(), with 600 inputs instead of 1200
torch.manual_seed(42)
model = Classifier(600, 2, inner_dim=inner_dim).to(device)
optimizer = Adam(model.parameters(), lr=learning_rate)
best_valid_loss = float("inf")
# number of epochs in a row without a new lowest validation loss
epochs_without_improvement = 0

for epoch in range(num_epochs):
    model.train()
    for x, y in train_loader_hyp:
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss, train_acc = evaluate(model, train_eval_loader_hyp)
    valid_loss, valid_acc = evaluate(model, valid_loader_hyp)
    # print every 10th epoch (all epochs are in W&B)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, train acc {train_acc:.2%}, valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")
    wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_valid_acc = valid_acc
        best_epoch = epoch + 1
        torch.save(model.state_dict(), checkpoint_path)
        epochs_without_improvement = 0
    else:
        # early stopping: no new lowest validation loss for `patience` epochs in a row
        epochs_without_improvement += 1
        if epochs_without_improvement == patience:
            print(f"Early stopping: no lower validation loss in the last {patience} epochs")
            break

wandb.run.summary["best_val_loss"] = best_valid_loss
wandb.run.summary["best_val_acc"] = best_valid_acc
wandb.run.summary["best_epoch"] = best_epoch
wandb.finish()
print(f"Best epoch: {best_epoch}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")
print("For comparison, all four texts (best run of the grid): valid loss 0.6492, valid acc 62.30%")

Epoch 10: train loss 0.6762, train acc 60.54%, valid loss 0.6799, valid acc 58.00%
Epoch 20: train loss 0.6561, train acc 62.63%, valid loss 0.6650, valid acc 59.30%
Epoch 30: train loss 0.6412, train acc 63.91%, valid loss 0.6552, valid acc 60.10%
Epoch 40: train loss 0.6304, train acc 65.16%, valid loss 0.6493, valid acc 61.40%
Epoch 50: train loss 0.6216, train acc 65.94%, valid loss 0.6461, valid acc 62.60%
Epoch 60: train loss 0.6147, train acc 66.71%, valid loss 0.6427, valid acc 63.50%
Epoch 70: train loss 0.6086, train acc 67.20%, valid loss 0.6415, valid acc 63.60%
Epoch 80: train loss 0.6030, train acc 67.67%, valid loss 0.6400, valid acc 64.30%
Epoch 90: train loss 0.5978, train acc 68.14%, valid loss 0.6394, valid acc 64.70%
Epoch 100: train loss 0.5931, train acc 68.59%, valid loss 0.6383, valid acc 65.00%
Best epoch: 100, valid loss 0.6383, valid acc 65.00%
For comparison, all four texts (best run of the grid): valid loss 0.6492, valid acc 62.30%


## Sweep
Question: **do finer learning rates, another number of hidden units or a smaller batch size lower the validation loss?** A W&B sweep (course slides "Project Discussions" (Week 7), slide 57) trains all 18 combinations: learning rate 3.16e-4, 1e-4 and 3.16e-5 (finer steps around the best value, slide 52), 100, 300 and 600 hidden units, and the largest batch size (slide 51) and 32, the batch size of the course exercise "Experiment Tracking".

The method is `grid` (slide 44), so that a new start tests the same configurations: `random` and `bayes` (slides 45 and 46) sample new values, and the W&B documentation lists no seed for them. Every run has seed 42, patience 10 and at most 300 epochs. One combination (300 hidden units, learning rate 1e-4, the largest batch size) repeats an earlier run and must give the same best epoch and validation loss. The selection rule stays: the run with the lowest validation loss of all candidate runs is chosen. Selecting among more runs also makes that validation loss more optimistic.

In [22]:
# the sweep: all combinations of these values (3 x 3 x 2 = 18 runs)
sweep_config = {
    "name": "model1-grid",
    "method": "grid",
    # the value in the run summary by which W&B compares the runs
    "metric": {"name": "best_val_loss", "goal": "minimize"},
    "parameters": {
        "learning_rate": {"values": [0.000316, 0.0001, 0.0000316]},
        "inner_dim": {"values": [100, 300, 600]},
        "batch_size": {"values": [MAX_BATCH, 32]},
    },
}

# one run of the sweep: the W&B agent calls this function once per combination
def sweep_run(max_epochs=300, patience=10, model_name="glove-mlp-sweep"):
    # the agent puts the values of this combination into run.config
    run = wandb.init(project="nlp-project1-art-gpu")
    learning_rate = run.config["learning_rate"]
    inner_dim = run.config["inner_dim"]
    batch_size = run.config["batch_size"]
    # run name as for the other runs, and the settings that are the same in all runs
    name = f"{model_name}-h{inner_dim}-lr{learning_rate}-b{batch_size}"
    run.name = name
    run.config.update({"model": model_name, "max_epochs": max_epochs, "patience": patience, "seed": 42})
    checkpoint_path = f"checkpoints/{name}.pth"
    # training batches with the batch size of this run
    sweep_train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True, drop_last=True)

    # same training as in train(), with early stopping
    torch.manual_seed(42)
    model = Classifier(1200, 2, inner_dim=inner_dim).to(device)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")
    # number of epochs in a row without a new lowest validation loss
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        for x, y in sweep_train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            # new lowest validation loss: save this epoch
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            # early stopping: no new lowest validation loss for `patience` epochs in a row
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                break

    # best values in the run summary, so that the runs can be sorted by them in W&B
    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    print(f"{name}: best epoch {best_epoch} of {epoch + 1}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")

In [ ]:
# register the sweep at W&B and train all its combinations (a grid ends by itself)
sweep_id = wandb.sweep(sweep_config, project="nlp-project1-art-gpu")
wandb.agent(sweep_id, function=sweep_run)


Create sweep with ID: f1gvuj2o
Sweep URL: https://app.forge.coreweave.com/maria-micioni-hochschule-luzern/nlp-project1-art-gpu/sweeps/f1gvuj2o
glove-mlp-sweep-h100-lr0.000316-b2048: best epoch 35 of 45, valid loss 0.6516, valid acc 61.60%
glove-mlp-sweep-h100-lr0.0001-b2048: best epoch 122 of 132, valid loss 0.6500, valid acc 62.00%
glove-mlp-sweep-h100-lr3.16e-05-b2048: best epoch 293 of 300, valid loss 0.6510, valid acc 61.80%
glove-mlp-sweep-h300-lr0.000316-b2048: best epoch 41 of 51, valid loss 0.6473, valid acc 63.00%
glove-mlp-sweep-h300-lr0.0001-b2048: best epoch 74 of 84, valid loss 0.6492, valid acc 62.30%
glove-mlp-sweep-h300-lr3.16e-05-b2048: best epoch 195 of 205, valid loss 0.6492, valid acc 61.60%
glove-mlp-sweep-h600-lr0.000316-b2048: best epoch 40 of 50, valid loss 0.6460, valid acc 64.00%
glove-mlp-sweep-h600-lr0.0001-b2048: best epoch 78 of 88, valid loss 0.6471, valid acc 63.70%
glove-mlp-sweep-h600-lr3.16e-05-b2048: best epoch 172 of 182, valid loss 0.6480, valid ac

In [23]:
# load the best checkpoint of every sweep run and sort by the validation loss, as in the section "Results"
sweep_values = sweep_config["parameters"]
results = []
for learning_rate in sweep_values["learning_rate"]["values"]:
    for inner_dim in sweep_values["inner_dim"]["values"]:
        for batch_size in sweep_values["batch_size"]["values"]:
            name = f"glove-mlp-sweep-h{inner_dim}-lr{learning_rate}-b{batch_size}"
            model = Classifier(1200, 2, inner_dim=inner_dim).to(device)
            model.load_state_dict(torch.load(f"checkpoints/{name}.pth", weights_only=True))
            valid_loss, valid_acc = evaluate(model, valid_loader)
            results.append((valid_loss, valid_acc, name))

for valid_loss, valid_acc, name in sorted(results):
    print(f"{name}: valid loss {valid_loss:.4f}, valid acc {valid_acc:.2%}")

# the other notebooks use the name and settings of the chosen run: stop if another run is now the lowest
assert sorted(results)[0][2] == "glove-mlp-sweep-h600-lr0.000316-b2048", f"another run is now the lowest: {sorted(results)[0]}"

glove-mlp-sweep-h600-lr0.000316-b2048: valid loss 0.6460, valid acc 64.00%
glove-mlp-sweep-h600-lr0.0001-b32: valid loss 0.6462, valid acc 62.60%
glove-mlp-sweep-h300-lr0.0001-b32: valid loss 0.6465, valid acc 62.80%
glove-mlp-sweep-h600-lr3.16e-05-b32: valid loss 0.6471, valid acc 63.10%
glove-mlp-sweep-h600-lr0.0001-b2048: valid loss 0.6471, valid acc 63.70%
glove-mlp-sweep-h300-lr0.000316-b2048: valid loss 0.6473, valid acc 63.00%
glove-mlp-sweep-h600-lr3.16e-05-b2048: valid loss 0.6480, valid acc 62.80%
glove-mlp-sweep-h600-lr0.000316-b32: valid loss 0.6481, valid acc 62.50%
glove-mlp-sweep-h100-lr0.0001-b32: valid loss 0.6487, valid acc 61.90%
glove-mlp-sweep-h300-lr0.0001-b2048: valid loss 0.6492, valid acc 62.30%
glove-mlp-sweep-h300-lr3.16e-05-b2048: valid loss 0.6492, valid acc 61.60%
glove-mlp-sweep-h300-lr3.16e-05-b32: valid loss 0.6495, valid acc 61.80%
glove-mlp-sweep-h100-lr0.000316-b32: valid loss 0.6500, valid acc 62.80%
glove-mlp-sweep-h100-lr0.0001-b2048: valid loss 0

The best sweep run has the largest learning rate and the most hidden units of the sweep, so the next values are tested too (course slides "Project Discussions" (Week 7), slide 53): 1000 hidden units and learning rate 0.001, which ended at its limit of 20 epochs in the grid and has at most 300 here, in 4 runs with the largest batch size. **Fixed before these runs:** they are a report, not candidates, and the chosen run stays whatever they show, because another value at the edge would start the next round and `04_model2_rnn.ipynb`, `04b_per_choice.ipynb` and `05_evaluation.ipynb` depend on the chosen run. They have their own names and checkpoints; one repeats the best sweep run as a check.

In [24]:
# the next values of the best sweep run: more hidden units and a larger learning rate, with the largest batch size
# a report, not candidates: own run names and checkpoints, so the checkpoint of the chosen run stays as it is
boundary_config = {
    "name": "model1-next-values",
    "method": "grid",
    "metric": {"name": "best_val_loss", "goal": "minimize"},
    "parameters": {
        "learning_rate": {"values": [0.000316, 0.001]},
        "inner_dim": {"values": [600, 1000]},
        "batch_size": {"values": [MAX_BATCH]},
    },
}
boundary_id = wandb.sweep(boundary_config, project="nlp-project1-art-gpu")
wandb.agent(boundary_id, function=lambda: sweep_run(model_name="glove-mlp-next-values"))

Create sweep with ID: h4bo53bt
Sweep URL: https://app.forge.coreweave.com/maria-micioni-hochschule-luzern/nlp-project1-art-gpu/sweeps/h4bo53bt
glove-mlp-next-values-h600-lr0.000316-b2048: best epoch 40 of 50, valid loss 0.6460, valid acc 64.00%
glove-mlp-next-values-h600-lr0.001-b2048: best epoch 28 of 38, valid loss 0.6471, valid acc 63.30%
glove-mlp-next-values-h1000-lr0.000316-b2048: best epoch 28 of 38, valid loss 0.6483, valid acc 62.90%
glove-mlp-next-values-h1000-lr0.001-b2048: best epoch 16 of 26, valid loss 0.6491, valid acc 62.00%


All runs use the same 1000 validation examples, so two runs can be compared with the exact McNemar test (McNemar, 1947): the chosen run against the best run of the grid (300 hidden units, learning rate 0.0001) and against the run that learned with the lowest accuracy (50 hidden units, learning rate 0.0001).

In [25]:
from scipy.stats import binomtest

# True where a saved run predicts the validation example correctly
def correct_predictions(name, inner_dim):
    model = Classifier(1200, 2, inner_dim=inner_dim).to(device)
    model.load_state_dict(torch.load(f"checkpoints/{name}.pth", weights_only=True))
    model.eval()
    with torch.no_grad():
        return model(X_valid).argmax(dim=1) == y_valid

chosen = correct_predictions("glove-mlp-sweep-h600-lr0.000316-b2048", 600)
for name, inner_dim in [("glove-mlp-h300-lr0.0001-e100", 300), ("glove-mlp-h50-lr0.0001-e100", 50)]:
    other = correct_predictions(name, inner_dim)
    # & = both True, ~ = not
    only_chosen = (chosen & ~other).sum().item()
    only_other = (~chosen & other).sum().item()
    # exact McNemar test: binomial test on the examples that only one of the two runs gets right
    p = binomtest(only_chosen, only_chosen + only_other, 0.5).pvalue
    print(f"chosen vs. {name}: accuracy {chosen.float().mean().item():.2%} vs. {other.float().mean().item():.2%}, only chosen right {only_chosen}, only other right {only_other}, p = {p:.4f}")

chosen vs. glove-mlp-h300-lr0.0001-e100: accuracy 64.00% vs. 62.30%, only chosen right 53, only other right 36, p = 0.0893
chosen vs. glove-mlp-h50-lr0.0001-e100: accuracy 64.00% vs. 60.70%, only chosen right 88, only other right 55, p = 0.0072


## Other seeds
Question: **how much do the validation scores change with the seed, and does the chosen run stay ahead of the best run of the grid?** Both configurations are trained with seeds 1 to 4, otherwise as in the sweep (course slides "Project Discussions" (Week 7), slide 42: the average of several runs shows how robust a model is). The seed is not tuned (slide 56): the chosen checkpoint stays the one with seed 42, and these runs are only reported.

In [26]:
# the two configurations: hidden units, learning rate, and the checkpoint name of the run with seed 42
seed_configurations = {
    "chosen": (600, 0.000316, "glove-mlp-sweep-h600-lr0.000316-b2048"),
    "best-of-grid": (300, 0.0001, "glove-mlp-h300-lr0.0001-e100"),
}

seed_config = {
    "name": "model1-seeds",
    "method": "grid",
    "metric": {"name": "best_val_loss", "goal": "minimize"},
    "parameters": {
        "configuration": {"values": list(seed_configurations)},
        "seed": {"values": [1, 2, 3, 4]},
    },
}

# one run: as in sweep_run(), with the seed of this run
def seed_run(batch_size=MAX_BATCH, max_epochs=300, patience=10):
    run = wandb.init(project="nlp-project1-art-gpu")
    configuration = run.config["configuration"]
    seed = run.config["seed"]
    inner_dim, learning_rate, _ = seed_configurations[configuration]
    name = f"glove-mlp-seed{seed}-h{inner_dim}-lr{learning_rate}"
    run.name = name
    run.config.update({"model": "glove-mlp-seeds", "inner_dim": inner_dim, "learning_rate": learning_rate, "batch_size": batch_size, "max_epochs": max_epochs, "patience": patience})
    checkpoint_path = f"checkpoints/{name}.pth"
    seed_train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True, drop_last=True)

    # the seed of this run instead of 42
    torch.manual_seed(seed)
    model = Classifier(1200, 2, inner_dim=inner_dim).to(device)
    optimizer = Adam(model.parameters(), lr=learning_rate)
    best_valid_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()
        for x, y in seed_train_loader:
            output = model(x)
            loss = criterion(output, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate(model, train_eval_loader)
        valid_loss, valid_acc = evaluate(model, valid_loader)
        wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_valid_acc = valid_acc
            best_epoch = epoch + 1
            torch.save(model.state_dict(), checkpoint_path)
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement == patience:
                break

    wandb.run.summary["best_val_loss"] = best_valid_loss
    wandb.run.summary["best_val_acc"] = best_valid_acc
    wandb.run.summary["best_epoch"] = best_epoch
    wandb.finish()
    print(f"{name}: best epoch {best_epoch} of {epoch + 1}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")

seed_id = wandb.sweep(seed_config, project="nlp-project1-art-gpu")
wandb.agent(seed_id, function=seed_run)

Create sweep with ID: s5lyls39
Sweep URL: https://app.forge.coreweave.com/maria-micioni-hochschule-luzern/nlp-project1-art-gpu/sweeps/s5lyls39
glove-mlp-seed1-h600-lr0.000316: best epoch 26 of 36, valid loss 0.6471, valid acc 63.00%
glove-mlp-seed2-h600-lr0.000316: best epoch 36 of 46, valid loss 0.6462, valid acc 62.80%
glove-mlp-seed3-h600-lr0.000316: best epoch 33 of 43, valid loss 0.6471, valid acc 63.20%
glove-mlp-seed4-h600-lr0.000316: best epoch 37 of 47, valid loss 0.6451, valid acc 62.80%
glove-mlp-seed1-h300-lr0.0001: best epoch 77 of 87, valid loss 0.6479, valid acc 63.00%
glove-mlp-seed2-h300-lr0.0001: best epoch 68 of 78, valid loss 0.6493, valid acc 62.70%
glove-mlp-seed3-h300-lr0.0001: best epoch 98 of 108, valid loss 0.6479, valid acc 64.00%
glove-mlp-seed4-h300-lr0.0001: best epoch 78 of 88, valid loss 0.6495, valid acc 62.50%


In [27]:
# validation loss and accuracy of all five seeds, from the saved checkpoints
for configuration, (inner_dim, learning_rate, seed42_name) in seed_configurations.items():
    names = [seed42_name] + [f"glove-mlp-seed{seed}-h{inner_dim}-lr{learning_rate}" for seed in [1, 2, 3, 4]]
    scores = []
    for name in names:
        model = Classifier(1200, 2, inner_dim=inner_dim).to(device)
        model.load_state_dict(torch.load(f"checkpoints/{name}.pth", weights_only=True))
        scores.append(evaluate(model, valid_loader))
    losses, accuracies = zip(*scores)
    # ddof=1: sample standard deviation of the five runs
    print(f"{configuration}: valid loss {np.mean(losses):.4f} (sd {np.std(losses, ddof=1):.4f}, seeds {[round(float(v), 4) for v in losses]}), valid acc {np.mean(accuracies):.2%} (sd {np.std(accuracies, ddof=1):.2%}, seeds {[round(v * 100, 2) for v in accuracies]})")


chosen: valid loss 0.6463 (sd 0.0009, seeds [0.646, 0.6471, 0.6462, 0.6471, 0.6451]), valid acc 63.16% (sd 0.50%, seeds [64.0, 63.0, 62.8, 63.2, 62.8])
best-of-grid: valid loss 0.6488 (sd 0.0008, seeds [0.6492, 0.6479, 0.6493, 0.6479, 0.6495]), valid acc 62.90% (sd 0.67%, seeds [62.3, 63.0, 62.7, 64.0, 62.5])


## Conclusion
Unless stated, one run per configuration at seed 42.

- **Chosen:** 600 hidden units, learning rate 0.000316, batch size 2048, best epoch 40: validation loss 0.6460, accuracy 64.00%, the lowest validation loss of all candidate runs.
- **All candidate runs that learned are close:** validation loss 0.6460 to 0.6567, accuracy 60.70% to 64.00%.
- **Exact McNemar test:** the chosen run is not significantly better than the best run of the grid (64.00% vs. 62.30%, only chosen right 53, only other right 36, p = 0.0893), but it is better than the run that learned with the lowest accuracy (64.00% vs. 60.70%, 88 and 55, p = 0.0072).
- **Other seeds:** over 5 seeds, the chosen configuration has a lower validation loss than the best configuration of the grid (mean 0.6463, sd 0.0009, vs. 0.6488, sd 0.0008), and all its seeds are below all of the other's. The accuracy does not separate them (63.16%, sd 0.50%, vs. 62.90%, sd 0.67%), and 64.00% is the highest of its five seeds.
- **Learning rate:** 0.01 can collapse training: with 50 hidden units, no unit stays active and the model predicts h1 for every example (50.80%). With 100 hidden units, more epochs do not help 0.01 (same best epoch 20, 0.6524).
- **Active units:** with learning rate 0.01, only 3 of 100 and 8 of 300 units are active, yet these runs reach 61.20% and 62.00%. The experiments do not show why.
- **Epoch limit:** 8 grid runs end at or near their limit (for example 300 hidden units, learning rate 0.00001: best epoch 300 of 300), so they may be underrated.
- **Batch size:** at the same settings, 32 has the lower validation loss in 6 of 9 cases and 2048 in 3, by at most 0.0059; the best runs are level (2048: 0.6460, 32: 0.6462).
- **Layer normalization:** no lower validation loss (0.6525 vs. 0.6492), with one learning rate and one seed.
- **Sweep:** finer learning rates, 600 hidden units and batch size 32 change little (0.6460 to 0.6532). The repeated run gives the same result.
- **Next values** (a report, not candidates): none is below the chosen run. 600 hidden units with learning rate 0.001: 0.6471; 1000 with 0.000316: 0.6483; 1000 with 0.001: 0.6491. The repeat of the chosen run gives the same result (best epoch 40, 0.6460).
- **Hypothesis-only:** a lower validation loss (0.6383 vs. 0.6492) and a higher accuracy (65.00% vs. 62.30%) than all four texts with the same settings, at its limit of 100 epochs, so in this format the observations show no measurable benefit on the validation set. It is unfiltered and the test set adversarially filtered (Bhagavatula et al., 2020; see "Data" in `README.md`), so this may not carry over.
- **CPU run:** the settings chosen in the CPU run (100 hidden units, learning rate 0.000316, batch size 32, seed 42) reach 0.6445 on the laptop and 0.6500 here, about half the range of all candidate runs. The two runs also differ slightly in tokenization and software, so this comparison does not isolate the effect of hardware; the chosen configuration is still not clearly the best one.
- **Input format:** `per-choice` is tested in `04b_per_choice.ipynb`; the selection stays within the fixed-position format.

## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Loading the features: `np.load` as in the NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.savez.html.
- Tensors: `torch.tensor` as in the course exercise "PyTorch" (Week 1).
- Naming: `X` (features, uppercase because it is a matrix) and `y` (labels) following the scikit-learn convention, https://scikit-learn.org/stable/glossary.html.
- Classifier: class `MyNetwork` from the course exercise "PyTorch" (Week 1), renamed to `Classifier`, based on the PyTorch tutorial "Neural Networks", https://docs.pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html; architecture "linear layer → ReLU → linear layer" from the course slides "Preliminaries" (Week 1), slide 25.
- Parameters and test forward pass: `named_parameters()` and a forward pass with a small batch as in the course exercise "PyTorch" (Week 1).
- Loss: cross-entropy from the course slides "Preliminaries" (Week 1), slide 27; `criterion` and `loss = criterion(output, target)` as in the course exercise "PyTorch" (Week 1); `nn.CrossEntropyLoss` (applies the softmax to the logits) from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html; the term "logits" from the course slides "Project Discussions" (Week 5), slide 27.
- Random seed: fixed seed 42 as in the course exercise "Experiment Tracking" (`seed_everything(42)`); `torch.manual_seed` from the PyTorch documentation "Reproducibility", https://docs.pytorch.org/docs/stable/notes/randomness.html.
- Optimizer and training step: Adam from the course slides "Preliminaries" (Week 1), slide 29; `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()` as in the course exercise "PyTorch" (Week 1).
- Batches: `DataLoader` with `batch_size=32` and `shuffle=True` for training and no shuffling for validation as in the course exercise "Experiment Tracking"; `TensorDataset` and `drop_last=True` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/data.html; `next(iter(...))` as in the PyTorch tutorial "Datasets & DataLoaders", https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html.
- Evaluation: collecting the batch losses, `argmax` predictions and labels, joining them with `torch.cat` and averaging the batch losses as in the validation of the course exercise "Experiment Tracking" (`validation_step` and `on_validation_epoch_end`); `model.eval()` and `torch.no_grad()` as in `test_loop` of the PyTorch tutorial "Optimizing Model Parameters", https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html; `np.mean` from NumPy (Harris et al., 2020).
- Training loop: epochs with a train loop over all batches and an evaluation after each epoch, and `model.train()`, as in the PyTorch tutorial "Optimizing Model Parameters"; the training step inside as in the course exercise "PyTorch" (Week 1); training as a function `train()` with the hyperparameters as arguments (with default values) and the seed set inside, as in the course exercise "Experiment Tracking".
- Experiment tracking: `wandb.login()`, `wandb.init(project=..., config=...)` and `wandb.finish()` as in the course exercise "Experiment Tracking"; run name `name=` from the W&B `wandb.init()` reference, https://docs.wandb.ai/models/ref/python/functions/init; logging the metrics once per epoch with `epoch` as x-axis as in the W&B documentation "Log objects and media", https://docs.wandb.ai/models/track/log (written there as `run.log`, which is the same as `wandb.log` for the current run); `train_loader.batch_size` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/data.html; Weights and Biases (Biewald, 2020); `nbformat` installed for W&B, as in the course exercise "Experiment Tracking".
- Saving the best model: `torch.save(model.state_dict(), ...)` and `model.load_state_dict(torch.load(..., weights_only=True))` as in the PyTorch tutorial "Save and Load the Model", https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html; `os.makedirs` from the Python standard library, https://docs.python.org/3/library/os.html; `float("inf")` from the Python documentation, https://docs.python.org/3/library/functions.html#float; best values in the run summary (`run.summary[...]`, here `wandb.run.summary` for the current run) as in the W&B documentation "Log summary metrics", https://docs.wandb.ai/models/track/log/log-summary.
- Experiments: hyperparameter tuning on the validation set, as on slide 10 of the course slides "Course Project HS26" (compare model settings, select the best model and hyperparameters).
- Results: loading the checkpoints with `load_state_dict` as in the PyTorch tutorial "Save and Load the Model"; sorting the results with `sorted()` (tuples are compared by their first item first) as in the Python documentation "Sorting Techniques", https://docs.python.org/3/howto/sorting.html.
- Active ReLU units: term and explanation "dying ReLU" from Lu, Shin, Su & Karniadakis (2020), *Dying ReLU and Initialization: Theory and Numerical Examples*, Communications in Computational Physics, https://doi.org/10.4208/cicp.OA-2020-0165; `torch.no_grad()` as in the PyTorch tutorial "Optimizing Model Parameters"; share of examples per unit with `.float().mean(dim=0)` as in the evaluation (`torch.mean`, PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.mean.html); counting the active units with `.sum().item()` as in `test_loop` of the PyTorch tutorial "Optimizing Model Parameters".
- Training decisions: course slides "Project Discussions": 1 output with a sigmoid or 2 outputs with a softmax (Week 5, slide 22), "double sigmoid" (Week 6, slide 30), Adam and AdamW (Week 6, slide 31), the learning rate as the most important hyperparameter (Week 6, slide 32), a constant learning rate (Week 6, slide 33, citing Hägele, A., Bakouch, E., Kosson, A., Ben Allal, L., Von Werra, L., & Jaggi, M. (2024), *Scaling Laws and Compute-Optimal Training Beyond Fixed Training Durations*, https://arxiv.org/abs/2405.18392), regularization (Week 6, slide 34), early stopping (Week 6, slide 38), batch size (Week 4, slide 14). AdamW: algorithm in the PyTorch documentation `torch.optim.AdamW`, https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html, from Loshchilov & Hutter (2019), *Decoupled Weight Decay Regularization*, ICLR 2019, https://arxiv.org/abs/1711.05101.
- Grid search, learning rate 0.0001, learning rate 0.00001 and 1000 hidden units: grid search (all combinations) from the course slides "Project Discussions" (Week 7), slide 44; testing beyond the boundary of the tested range (slide 53), exponentially changing values (slide 52) and a sensible number of epochs (slide 41) from the same slides.
- Hypothesis-only check: hypothesis-only baselines from Gururangan, S., Swayamdipta, S., Levy, O., Schwartz, R., Bowman, S., & Smith, N. A. (2018), *Annotation Artifacts in Natural Language Inference Data*, NAACL 2018, https://arxiv.org/abs/1803.02324, and Poliak, A., Naradowsky, J., Haldar, A., Rudinger, R., & Van Durme, B. (2018), *Hypothesis Only Baselines in Natural Language Inference*, *SEM 2018, https://arxiv.org/abs/1805.01042; for ART: Bhagavatula et al. (2020), Table 1 (BERT-ft [h_i Only], 59.5%); column indexing `[:, ...]` (here `X_train[:, 600:]`) as in the course exercise "PyTorch" (Week 1); training steps as in `train()` above; origin of the splits: Bhagavatula et al. (2020), Table 6, and course slides "Course Project HS26", slide 8.
- GPU: `torch.device`, `torch.cuda.is_available` and `Tensor.to`, PyTorch documentation "CUDA semantics", https://docs.pytorch.org/docs/stable/notes/cuda.html; `torch.cuda.empty_cache` and `torch.cuda.OutOfMemoryError`, https://docs.pytorch.org/docs/stable/cuda.html; `Tensor.repeat`, https://docs.pytorch.org/docs/stable/generated/torch.Tensor.repeat.html. HSLU GPUHub and Google Colab from the course slides "Course Project HS26", slide 11. The largest batch size, found with the longest example, one forward and backward pass and binary search, from the course slides "Project Discussions" (Week 4), slide 14, and (Week 7), slide 51. Environment variables on Colab: Google Colab Secrets and `drive.mount`, https://colab.research.google.com/. Repeated runs on a GPU: `torch.backends.cudnn.deterministic`, https://docs.pytorch.org/docs/stable/backends.html, and PyTorch documentation "Reproducibility", https://docs.pytorch.org/docs/stable/notes/randomness.html; `torch.use_deterministic_algorithms` (`warn_only`), https://docs.pytorch.org/docs/stable/generated/torch.use_deterministic_algorithms.html; `CUBLAS_WORKSPACE_CONFIG`, from the warning about RNN functions in `torch.nn.LSTM`, https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html. Setup on Colab: `sys.modules`, https://docs.python.org/3/library/sys.html#sys.modules, and `os.chdir`, https://docs.python.org/3/library/os.html#os.chdir, Python documentation; `%pip`, IPython documentation, https://ipython.readthedocs.io/en/stable/interactive/magics.html.
- Tools: Claude (Anthropic), ChatGPT (OpenAI) and Gemini (Google), see "Use of AI" in README.md; all code reviewed and understood by me.
- Layer normalization: `nn.LayerNorm` from the PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html; placed between the linear layer and the ReLU following Ba, Kiros & Hinton (2016), *Layer Normalization* ("after the normalization but before the non-linearity"), https://arxiv.org/abs/1607.06450; normalization as an option and layer norm (yes/no) as an experiment from the course slides "Project Discussions" (Week 6), slide 34, and (Week 7), slide 44. Check: `torch.equal`, https://docs.pytorch.org/docs/stable/generated/torch.equal.html, and `torch.numel`, https://docs.pytorch.org/docs/stable/generated/torch.numel.html, from the PyTorch documentation.
- Sweep: W&B sweeps from the course slides "Project Discussions" (Week 7), slide 57, and the W&B documentation on the sweep configuration, https://docs.coreweave.com/models/sweeps/define-sweep-configuration, its options (`method`, `metric`, `parameters`, `values`), https://docs.coreweave.com/models/sweeps/sweep-config-keys, and the sweeps walkthrough (`wandb.sweep`, `wandb.agent` with `function`, `wandb.init` inside the function, `run.config`), https://docs.coreweave.com/models/sweeps/walkthrough; `run.config.update`, W&B documentation on the run config, https://docs.coreweave.com/models/track/config; `run.name`, W&B documentation `Run`, https://docs.coreweave.com/models/ref/python/experiments/run; grid search (slide 44), random search (slide 45), Bayesian optimization (slide 46), finer steps such as 1e-4.5 (slide 52), the next value beyond the boundary of the tested range (slide 53) and the batch size (slide 51) from the same slides; early stopping with a patience from the same slides (Week 6), slide 38. `lambda`, Python tutorial, https://docs.python.org/3/tutorial/controlflow.html#lambda-expressions (also in the section "Batches"); the `assert` statement, Python documentation, https://docs.python.org/3/reference/simple_stmts.html#the-assert-statement. Comparison of two runs: exact McNemar test (McNemar, 1947) with `scipy.stats.binomtest` as in `04_model2_rnn.ipynb` (see the sources there), https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binomtest.html.
- Other seeds: reporting the average of several runs and the debate about tuning the seed from the course slides "Project Discussions" (Week 7), slides 42 and 56; `np.std` with `ddof`, NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.std.html; `zip(*scores)` and `round()`, Python documentation, https://docs.python.org/3/library/functions.html; `list()` of a dictionary (its keys), Python tutorial "Dictionaries", https://docs.python.org/3/tutorial/datastructures.html#dictionaries.
- Output of W&B: `WANDB_SILENT` (no W&B log lines in the notebook output), W&B documentation "Environment variables", https://docs.coreweave.com/models/track/environment-variables; `os.environ`, Python documentation, https://docs.python.org/3/library/os.html#os.environ.